# PyTorch Tensor 基础 API 全面介绍

## 1. 环境初始化与依赖导入

In [ ]:
import torch  # 导入 PyTorch 深度学习框架，提供张量计算与自动微分功能
print(torch.__version__)  # 打印 PyTorch 版本号，确认安装正确；返回值：str，例如 '2.5.1+cpu'

import numpy as np  # 导入 NumPy 科学计算库，用于数组操作及与 Tensor 的相互转换

2.5.1+cpu


## 2. Tensor 基础概念

Tensor（张量）是 PyTorch 最核心的数据结构，类似于 NumPy 的 ndarray，
但额外支持 **GPU 加速**和**自动微分**。

| 阶数 | 名称 | 描述 | 示例 |
|------|------|------|------|
| 0 阶 | 标量（Scalar） | 单个数值 | `torch.tensor(3.14)` |
| 1 阶 | 向量（Vector） | 一维数组 | `torch.tensor([1, 2, 3])` |
| 2 阶 | 矩阵（Matrix） | 二维表格 | `torch.tensor([[1,2],[3,4]])` |
| ≥3 阶 | 高维张量 | 如图像批次 | `shape = [batch, C, H, W]` |

**本章主要内容：**
1. 创建张量的三种方式
2. 张量的常用方法与属性（`item`、`view`、`permute` 等）
3. 张量的索引、切片与数学运算
4. GPU 上的张量操作

## 3. 创建 Tensor 的三种方法

### 3.1 使用列表与 NumPy 数组创建 Tensor

In [ ]:
# 方式一：使用 Python 列表创建一维 Tensor
t1 = torch.Tensor([1, 2, 3])  # torch.Tensor() 接受 Python 列表，默认 float32 类型；返回 torch.FloatTensor
print(t1)  # 输出：tensor([1., 2., 3.])，注意整数列表被自动转为浮点数

# 方式二：使用 NumPy 数组创建二维 Tensor
array1 = np.arange(12).reshape(3, 4)  # 生成 0~11 整数，重塑为 3×4 矩阵；返回 np.ndarray, shape=(3,4)
t2 = torch.Tensor(array1)  # 将 NumPy ndarray 转为 float32 Tensor；返回 torch.FloatTensor, shape=[3,4]
print(t2)  # 输出 3×4 的浮点张量

tensor([1., 2., 3.])
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])


### 3.2 Tensor 的切片操作

In [ ]:
t2[0:1, :]  # 切片：取行索引 [0,1)（即第 0 行），所有列（:）
            # 语法与 NumPy 完全相同；返回形状为 [1, 4] 的二维 Tensor（保留维度）

tensor([[0., 1., 2., 3.]])

### 3.3 通过 torch 内置 API 创建特殊 Tensor

In [ ]:
"""
torch 提供多种创建特殊初始化 Tensor 的 API：
  torch.empty(3, 4)                         : 创建 3×4 未初始化 Tensor，元素值不确定（内存旧数据）
  torch.ones([3, 4])                        : 创建 3×4 全 1 浮点 Tensor
  torch.zeros([3, 4])                       : 创建 3×4 全 0 浮点 Tensor
  torch.rand([3, 4])                        : 创建 3×4 均匀分布随机 Tensor，取值范围 [0, 1)
  torch.randint(low=0, high=10, size=[3, 4]): 创建整数随机 Tensor，取值范围 [low, high)
  torch.randn([3, 4])                       : 创建标准正态分布 Tensor，均值=0，方差=1
"""
print(torch.empty(3, 4))    # 未初始化张量，内容不确定（通常显示 0，但不保证）
print(torch.ones([3, 4]))   # 全 1 张量，常用于权重初始化、掩码等场景
print(torch.zeros([3, 4]))  # 全 0 张量，常用于偏置初始化、累加器等场景
print(torch.rand([3, 4]))   # [0,1) 均匀分布随机张量，常用于测试与参数初始化

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
tensor([[0.4181, 0.2008, 0.9539, 0.9866],
        [0.9802, 0.6692, 0.8374, 0.9207],
        [0.2268, 0.9637, 0.0757, 0.6726]])


## 4. Tensor 的方法与属性

### 4.1 item()：从单元素 Tensor 中提取 Python 标量

In [ ]:
# item() 方法将只含一个元素的 Tensor 转换为 Python 原生数值
# 若 Tensor 含多个元素，调用 item() 会抛出 RuntimeError
a = torch.tensor(np.arange(1))  # 从 np.arange(1)=[0] 创建 Tensor；shape=(1,)，dtype=torch.int32
print(a)           # 输出：tensor([0], dtype=torch.int32)
print(a.item())    # 提取唯一元素，返回 Python int 0；类型：Python int（非 Tensor）
print('-' * 50)    # 打印分隔线，用于区分不同代码块的输出
print(torch.Tensor([[[1]]]).item())  # 三维嵌套单元素 Tensor，item() 返回 Python float 1.0
print('-' * 50)    # 打印分隔线

tensor([0], dtype=torch.int32)
0
--------------------------------------------------
1.0
--------------------------------------------------


### 4.2 Tensor 与 NumPy 互转及形状查询

In [ ]:
# Tensor 转换为 NumPy 数组及多种形状查询方式
t2 = torch.Tensor([[3, 4]])   # 创建形状为 [1, 2] 的二维浮点张量
print(t2.numpy())             # .numpy()：将 CPU Tensor 转换为 np.ndarray；注意两者共享内存（浅拷贝）
print('-' * 50)               # 打印分隔线
print(t2.shape)               # .shape 属性：返回 torch.Size，描述各维度大小；等价于 .size()
print(t2.size())              # .size() 方法：与 .shape 完全等价，返回 torch.Size([1, 2])
print(t2.size(0))             # .size(dim)：返回指定维度 dim 的大小；dim=0 表示第 0 维，返回 1

[[3. 4.]]
--------------------------------------------------
torch.Size([1, 2])
torch.Size([1, 2])
1


In [ ]:
t2  # 直接输出 Tensor 对象；Jupyter Notebook 会自动调用 __repr__ 方法展示内容

tensor([[3., 4.]])

### 4.3 NumPy reshape 的浅拷贝特性验证

In [ ]:
# reshape 是浅拷贝：修改 reshape 后的数组会同步修改原数组
array1 = np.array([[1, 2, 3], [4, 5, 6]])  # 创建 2×3 的 NumPy 整数数组
print(id(array1))                           # 打印 array1 的 Python 对象内存地址（唯一标识符）
array2 = array1.reshape(3, 2)               # 重塑为 3×2；array2 与 array1 共享底层数据（浅拷贝）
print(id(array2))                           # array2 是新的 Python 对象（id 不同），但底层数据相同

1418787627696
1418771602704


In [ ]:
# 验证浅拷贝特性：修改 array2 会影响 array1
array2[0, 0] = 100   # 修改 array2 第 0 行第 0 列的值为 100
print(array1)        # array1 对应位置也变为 100，证明两者共享底层内存
print(array2)        # array2 中对应位置同样为 100

[[100   2   3]
 [  4   5   6]]
[[100   2]
 [  3   4]
 [  5   6]]


In [ ]:
array1.ndim  # .ndim 属性：返回数组的维数（轴数）；array1 是二维数组，返回 Python int 2

2

### 4.4 view()：Tensor 的浅拷贝形状修改

In [ ]:
# view() 类似 NumPy 的 reshape，是浅拷贝（共享底层存储），仅改变形状描述，不复制数据
t2 = torch.Tensor([[[3, 4]]])    # 创建形状为 [1, 1, 2] 的三维浮点张量
print(t2.size())                 # 输出当前形状：torch.Size([1, 1, 2])
print(t2.view([1, 2]))           # 重塑为 [1, 2]（1行2列）；参数类型：list 或 tuple
print(t2.view([2]))              # 重塑为一维张量，长度为 2

torch.Size([1, 1, 2])
tensor([[3., 4.]])
tensor([3., 4.])


In [ ]:
b = t2.view([2, -1])  # -1 表示该维度由总元素数量自动推算；总元素=2，2/2=1，结果形状 [2,1]
print(b)              # 输出重塑后的列向量：[[3.], [4.]]
print('-' * 50)       # 打印分隔线
# t2 和 b 是不同的 Python 对象（id 不同），但底层存储相同（浅拷贝验证）
print(f'id(t2)={id(t2)}, id(b)={id(b)}')  # 两者 id 不同，表示不同的 Tensor 封装对象
# 参考文档：https://pytorch.org/docs/stable/tensor_view.html
# untyped_storage() 返回底层存储对象，data_ptr() 返回底层数据的内存地址指针
t2.untyped_storage().untyped().data_ptr() == b.untyped_storage().untyped().data_ptr()  # True 表示共享内存

tensor([[3.],
        [4.]])
--------------------------------------------------
id(t2)=1418808680576,id(b)=1417876953024


True

In [ ]:
# 验证 view() 浅拷贝：修改 b 会同步修改 t2
b[0, 0] = 100  # 修改 b 第 0 行第 0 列为 100
print(b)       # b 中第 0 行第 0 列变为 100
print(t2)      # t2 对应位置也变为 100，证明共享底层数据

tensor([[100.],
        [  4.]])
tensor([[[100.,   4.]]])


### 4.5 获取维数、最大值、转置与轴置换

In [ ]:
# 常用 Tensor 属性与变换方法演示
print(t2.dim())   # .dim()：返回 Tensor 的维数（秩）；t2 形状 [1,1,2]，维数为 3；返回 Python int

print(t2.max())   # .max()：返回 Tensor 中所有元素的最大值；返回标量 Tensor

# 二维矩阵转置
t3 = torch.tensor([[1, 3, 4], [2, 4, 6]])  # 创建形状为 [2, 3] 的整数二维张量
print(t3)       # 输出原始 2×3 矩阵
print(t3.t())   # .t()：对二维 Tensor 进行转置，返回形状为 [3, 2] 的 Tensor；等价于 t3.T

3
tensor(100.)
tensor([[1, 3, 4],
        [2, 4, 6]])
tensor([[1, 2],
        [3, 4],
        [4, 6]])


In [ ]:
# 多维 Tensor 轴置换：permute() 与 transpose() 对比
t4 = torch.tensor(np.arange(24).reshape(2, 3, 4))  # 创建形状为 [2, 3, 4] 的三维整数张量
print(t4.shape)                        # 原始形状：torch.Size([2, 3, 4])
print('-' * 50)                        # 分隔线
print(t4.transpose(0, 1).shape)        # .transpose(dim0,dim1)：交换两个轴；[2,3,4] → [3,2,4]
print('-' * 50)                        # 分隔线
print(t4.permute(1, 0, 2).shape)       # .permute(dims)：按指定顺序重排所有轴；(1,0,2) → [3,2,4]
print('-' * 50)                        # 分隔线
print(t4.permute(1, 2, 0).shape)       # 轴顺序改为 1→0, 2→1, 0→2；结果形状 [3, 4, 2]
print('-' * 50)                        # 分隔线
print(t4.permute(2, 1, 0).shape)       # 完全反转所有轴顺序；结果形状 [4, 3, 2]

torch.Size([2, 3, 4])
--------------------------------------------------
torch.Size([3, 2, 4])
--------------------------------------------------
torch.Size([3, 2, 4])
--------------------------------------------------
torch.Size([3, 4, 2])
--------------------------------------------------
torch.Size([4, 3, 2])


In [ ]:
t4.dtype  # .dtype 属性：返回 Tensor 的数据类型；由 np.arange 整数数组创建，对应 torch.int32

torch.int32

### 4.6 原地修改操作（方法名末尾带 `_` 的方法）

In [ ]:
# PyTorch 中，方法名末尾带 _ 的方法会原地修改调用它的 Tensor（不创建新对象）
# 类似 Pandas 的 inplace=True，可节省内存，但操作不可逆
x = torch.tensor(np.arange(12).reshape(3, 4), dtype=torch.int8)  # 创建 3×4 的 int8 整数张量，值为 0~11
print(x)   # 输出原始张量
y = torch.ones([3, 4], dtype=torch.int64)  # 创建 3×4 的全 1 int64 张量
print(y)   # 输出全 1 张量

print('-' * 50)  # 打印分隔线
x.sub_(y)        # .sub_(other)：原地减法，直接修改 x（x -= y）；返回修改后的 x 本身
print(x)         # 输出原地减 1 后的张量

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]], dtype=torch.int8)
tensor([[1, 1, 1, 1],
        [1, 1, 1, 1],
        [1, 1, 1, 1]])
--------------------------------------------------
tensor([[-1,  0,  1,  2],
        [ 3,  4,  5,  6],
        [ 7,  8,  9, 10]], dtype=torch.int8)


## 5. Tensor 的索引与切片

In [ ]:
# Tensor 索引与切片语法与 NumPy 完全相同
t5 = torch.tensor(np.arange(12).reshape(3, 4))  # 创建 3×4 整数张量，值为 0~11
print(t5)            # 输出完整 3×4 张量
print(t5[1, 2])      # 单元素索引：第 1 行第 2 列；返回零维标量 Tensor，值为 6
print(t5[1])         # 行索引：取整个第 1 行；返回形状为 [4] 的一维 Tensor
print(t5[:, 1])      # 列索引：取所有行的第 1 列；返回形状为 [3] 的一维 Tensor
print(t5[1:3, 1:3])  # 二维切片：第 1~2 行、第 1~2 列的子矩阵；返回形状为 [2, 2] 的 Tensor
print(t5[1:3, :])    # 行切片：第 1~2 行的所有列；返回形状为 [2, 4] 的 Tensor

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]], dtype=torch.int32)
tensor(6, dtype=torch.int32)
tensor([4, 5, 6, 7], dtype=torch.int32)
tensor([1, 5, 9], dtype=torch.int32)
tensor([[ 5,  6],
        [ 9, 10]], dtype=torch.int32)
tensor([[ 4,  5,  6,  7],
        [ 8,  9, 10, 11]], dtype=torch.int32)


## 6. Tensor 的数学运算

### 6.1 MSE（均方误差）计算

In [ ]:
# MSE（Mean Squared Error，均方误差）= mean((y_pred - y_true)^2)
# 是回归任务最常用的损失函数，衡量预测值与真实值之间的平均平方误差
t6 = torch.tensor(np.arange(16).reshape(16, 1), dtype=torch.float32)      # 16×1 列向量，值 0~15，float32
t7 = torch.tensor(np.arange(16, 32).reshape(16, 1), dtype=torch.float32)  # 16×1 列向量，值 16~31，float32
print(t6)  # 输出第一个列向量
print(t7)  # 输出第二个列向量
((t6 - t7) ** 2).mean()  # 逐元素差的平方再取均值；差值恒为 -16，平方为 256，均值为 256.0

tensor([[ 0.],
        [ 1.],
        [ 2.],
        [ 3.],
        [ 4.],
        [ 5.],
        [ 6.],
        [ 7.],
        [ 8.],
        [ 9.],
        [10.],
        [11.],
        [12.],
        [13.],
        [14.],
        [15.]])
tensor([[16.],
        [17.],
        [18.],
        [19.],
        [20.],
        [21.],
        [22.],
        [23.],
        [24.],
        [25.],
        [26.],
        [27.],
        [28.],
        [29.],
        [30.],
        [31.]])


tensor(256.)

### 6.2 广播机制（Broadcasting）

In [ ]:
# 广播机制：形状不完全相同的 Tensor 之间运算时，PyTorch 自动沿尺寸为 1 的维度扩展
# 规则：从右侧对齐维度，尺寸为 1 的维度自动广播到匹配大小
t8 = torch.tensor(np.arange(12).reshape(3, 4), dtype=torch.float32)  # 3×4 浮点张量
t9 = torch.tensor(np.arange(3).reshape(3, 1), dtype=torch.float32)   # 3×1 浮点列向量
print(t8)  # 输出 3×4 张量
print(t9)  # 输出 3×1 列向量
t8 - t9    # 广播：t9 的第 1 维从 1 扩展为 4，对应行元素相减；结果形状为 [3, 4]

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
tensor([[0.],
        [1.],
        [2.]])


tensor([[0., 1., 2., 3.],
        [3., 4., 5., 6.],
        [6., 7., 8., 9.]])

## 7. GPU 上的 Tensor 操作

In [ ]:
"""
GPU Tensor 使用步骤：
1. 创建设备对象：torch.device('cpu') 或 torch.device('cuda:0')
   - 'cuda:0' 表示第一块 GPU，使用前需确认 torch.cuda.is_available() 返回 True
2. 调用 tensor.to(device) 将 Tensor 迁移到指定设备
3. 不同设备上的 Tensor 不能直接混合运算，会抛出 RuntimeError
"""
print(torch.cuda.is_available())    # 检测当前环境是否有可用 CUDA GPU；返回 bool
if torch.cuda.is_available():        # 仅当 GPU 可用时执行以下代码
    device = torch.device('cuda')    # 创建 CUDA 设备对象，默认使用第一块 GPU (cuda:0)
    y = torch.ones_like(x, device=device)  # 创建与 x 形状相同的全 1 Tensor，直接分配在 GPU 上
    x = x.to(device)                 # 将 CPU Tensor x 迁移到 GPU；返回 GPU 上的新 Tensor
    z = x + y                         # GPU 上的加法运算，运算在 GPU 完成
    print(z)                          # 输出 GPU 运算结果（Tensor 显示 device='cuda:0'）
    print(z.to('cpu', torch.double))  # 将结果迁回 CPU，同时转换为 float64 类型

device = torch.device('cpu')  # 创建 CPU 设备对象
x.to(device)                  # 将张量迁移回 CPU；.to(device) 返回目标设备上的 Tensor

False


tensor([[-1,  0,  1,  2],
        [ 3,  4,  5,  6],
        [ 7,  8,  9, 10]], dtype=torch.int8)

## 8. 逐元素运算与矩阵乘法

In [ ]:
# 逐元素乘法（Element-wise Multiplication / Hadamard 积）
# 两个形状完全相同的 Tensor 对应位置元素相乘，与线性代数矩阵乘法不同
t8 = torch.tensor(np.arange(12).reshape(3, 4), dtype=torch.float32)  # 3×4 浮点张量 A
t9 = torch.tensor(np.arange(12).reshape(3, 4), dtype=torch.float32)  # 3×4 浮点张量 B（与 A 内容相同）
t8 * t9  # * 运算符执行逐元素乘法；等价于 torch.mul(t8, t9)；返回形状为 [3, 4] 的张量

tensor([[  0.,   1.,   4.,   9.],
        [ 16.,  25.,  36.,  49.],
        [ 64.,  81., 100., 121.]])

In [ ]:
# 矩阵乘法（Matrix Multiplication）
# torch.mm(A, B) 要求 A、B 均为二维 Tensor，且 A.shape[1] == B.shape[0]
torch.mm(t8, t9.transpose(0, 1))  # t8 形状 [3,4]，t9.T 形状 [4,3]；结果为 [3,3] 的矩阵乘积
                                   # 等价于数学中的 A × B^T；与逐元素乘法 * 完全不同

tensor([[ 14.,  38.,  62.],
        [ 38., 126., 214.],
        [ 62., 214., 366.]])